# Camping Person Account export for Salesforce comparison

Reproduces the cleaning pipeline from `person_account_data_quality.ipynb` (which remains the
analysis / QA record) and exports **all rows** in the **standard import format**
(column set of `crm_imp_person_accounts`, as used by the investor imports) plus per-row flags:

- `data_issue` — non-person records (legal entities, organisations, households, placeholders)
- `contract_valid` — row passes the Person Account contract checks (pandera) and is not a duplicate
- `import_ready` — the agreed upload scope (Option 4)

The flag is called `exclude` while working in this notebook and is renamed to `data_issue`
on export, matching the investor staging convention and avoiding a reserved word in SQL.
All three flags are written as `1`/`0`, like every other boolean-ish column in the format.

Decisions encoded here:

- `source = 'camping'`, `source_origin = 'grubhof'`
- `camping_customer = 1`; hotel/residences/invest customer flags and
  `consent_central`/`consent_residences`/`consent_invest` are `0` (confirmed against sheet 1)
- `gender` derived from salutation (Mr. → male, Ms. → female), as sheet 1 did.
  Lowercase is correct: the investor inserts staged `male`/`female` and Salesforce
  normalised them to `Male`/`Female` on write (3,904 rows verified in prod).
- `email` is lowercased — step 2 of the house cleaning standard in
  `sql/Scripts/gms_all_profiles_cleaning.sql`. Matters because email is the only join key
  to Salesforce and is written to `PersonEmail` on every new account.
- `middle_name` stays empty — sheet 2's `NA_NAME3` holds academic titles (Dr., Ing., Mag.), not middle names
- `birth_place`, `nationality_country_code`, `state` stay empty — sheet 1's versions were
  broken copies of the mailing country
- Phones that are not possible numbers are **blanked instead of invalidating the row**
- Dedup on the four matching fields (`first_name`, `last_name`, `email`, `birth_date`)

Name capitalisation uses `nameparser`, which leaves mixed-case names untouched and only
rewrites all-upper / all-lower ones — the same policy as Oleg's `fn_title_case` guard
(`WHERE fname = UPPER(fname) OR fname = LOWER(fname)`).

No `_`-prefixed pipeline columns are exported: `_operation`, `_batch_id`, `_excluded` and the
`_*_processed_at` timestamps are state owned by `crm_imp_person_accounts`, set by the batch
SQL and the bulk scripts. The investor staging table does not carry them either.

Filtering happens later in MySQL, mirroring the investor import flow (`data_issue` pattern).

In [ ]:
import re
from datetime import date
from pathlib import Path

import pandera.polars as pa
import phonenumbers
import polars as pl
import polars.selectors as cs
import pycountry
from nameparser import HumanName
from pandera.polars import PolarsData

WORKBOOK_PATH = "data/source/2607_Data cleaned - Grubhof.xlsx"
EXPORT_PATH = Path("../local_data/csv/20260807_camping_grubhof_cleaned.csv")

In [ ]:
COLUMN_RENAMES = {
    "NA_KURZ": "external_id",
    "NA_ANREDE": "salutation",
    "NA_NAME2": "first_name",
    "NA_NAME3": "middle_name",
    "NA_NAME1": "last_name",
    "NA_GEBDAT": "birth_date",
    "NA_TELEX": "email",
    "NA_TEL1": "phone",
    "NZ_SPRACHE": "preferred_language",
    "NA_STR": "address",
    "NA_CPLZ": "postal_code",
    "NA_ORT": "city",
    "NA_INT": "country",
    "NA_MAILING": "consent_camping",
}

source_data = pl.read_excel(
    WORKBOOK_PATH,
    sheet_id=2,
)
raw = (
    source_data
    .rename(COLUMN_RENAMES)
    .select(COLUMN_RENAMES.values())
)
profiled = raw.with_columns(cs.string().replace("", None))

blank_row = pl.all_horizontal(pl.all().is_null())
data = profiled.filter(~blank_row).drop("middle_name")

print(f"Data rows: {data.height:,}")

In [ ]:
transformed = data.with_columns(
    cs.string().str.strip_chars().replace("", None)
)

In [ ]:
emails_before = transformed.get_column("email")

transformed = transformed.with_columns(
    pl.col("email").str.to_lowercase()
)

print(
    "Emails lowercased: "
    f"{(emails_before != transformed['email']).sum():,}"
)

In [ ]:
LEGAL_ENTITY_PATTERN = (
    r"(?i)(?:"
    r"\bgmbh\b|\bco\.?\s*kg\b|\bltd\.?|\blimited\b|\bllc\b|"
    r"\binc\.?|\bag\b|\bkg\b|\bog\b|"
    r"\bb\.?\s*v\.?(?:\s|$)|\be\.?\s*v\.?(?:\s|$)"
    r")"
)
ORGANISATION_PATTERN = (
    r"(?i)\b(?:"
    r"verein|verband|club|stiftung|gemeinde|reisebüro|"
    r"travel\s+group|camping|caravan|hotel|verlag|touristik|union"
    r")\b"
)
HOUSEHOLD_PATTERN = (
    r"(?i)(?:"
    r"\bfamilie\b|\bfamily\b|\bfam\.?(?:\s|$)|\beheleute\b"
    r")"
)
NAME_PLACEHOLDERS = [
    "test",
    "unknown",
    "n/a",
    "none",
    "na",
    "null",
    "xxx",
    "first",
    "firstname",
    "first name",
    "last",
    "lastname",
    "last name",
    "-",
    ".",
]

full_name = pl.concat_str(
    "first_name",
    "last_name",
    separator=" ",
    ignore_nulls=True,
).str.strip_chars()

name_exclusion_flags = (
    transformed
    .select(
        "external_id",
        "first_name",
        "last_name",
        full_name.alias("full_name"),
    )
    .with_columns(
        pl.col("full_name")
        .str.contains(LEGAL_ENTITY_PATTERN)
        .alias("legal_entity"),
        pl.col("full_name")
        .str.contains(ORGANISATION_PATTERN)
        .alias("organisation"),
        pl.col("full_name")
        .str.contains(HOUSEHOLD_PATTERN)
        .alias("household"),
        pl.any_horizontal(
            pl.col("first_name").str.to_lowercase().is_in(NAME_PLACEHOLDERS),
            pl.col("last_name").str.to_lowercase().is_in(NAME_PLACEHOLDERS),
        ).alias("placeholder"),
        pl.col("full_name").str.contains(r"\d").alias("contains_digit"),
    )
    .with_columns(
        pl.any_horizontal(cs.boolean())
        .fill_null(False)
        .alias("exclude")
    )
)

transformed = transformed.with_columns(
    name_exclusion_flags.get_column("exclude")
)

print(
    "Rows flagged for exclusion: "
    f"{transformed.get_column('exclude').sum():,}"
)

In [ ]:
# nameparser only treats spaces as word boundaries, so all-caps initials come back
# lowercased: P.H.B. -> P.h.b. Oleg's fn_title_case treats '.' as a separator too.
LETTER_AFTER_DOT = re.compile(r"\.(\s*)([a-z])")


def capitalize_after_dots(value: str) -> str:
    return LETTER_AFTER_DOT.sub(
        lambda match: f".{match.group(1)}{match.group(2).upper()}",
        value,
    )


def capitalize_first_name(value: str) -> str:
    name = HumanName(first=value)
    name.capitalize()
    return capitalize_after_dots(name.first)


def capitalize_last_name(value: str) -> str:
    name = HumanName(last=value)
    name.capitalize()
    return capitalize_after_dots(name.last)


names_before = transformed.select(
    "first_name",
    "last_name",
)

transformed = transformed.with_columns(
    pl.when(~pl.col("exclude"))
    .then(
        pl.col("first_name").map_elements(
            capitalize_first_name,
            return_dtype=pl.String,
        )
    )
    .otherwise(pl.col("first_name"))
    .alias("first_name"),
    pl.when(~pl.col("exclude"))
    .then(
        pl.col("last_name").map_elements(
            capitalize_last_name,
            return_dtype=pl.String,
        )
    )
    .otherwise(pl.col("last_name"))
    .alias("last_name"),
)

lowercase_after_dot = pl.any_horizontal(
    pl.col("first_name").str.contains(r"\.[a-z]"),
    pl.col("last_name").str.contains(r"\.[a-z]"),
)

print(
    "First names capitalized: "
    f"{(names_before['first_name'] != transformed['first_name']).sum():,}"
)
print(
    "Last names capitalized: "
    f"{(names_before['last_name'] != transformed['last_name']).sum():,}"
)
print(
    "Initials left lowercase after a dot: "
    f"{transformed.filter(lowercase_after_dot).height:,}"
)

In [ ]:
PLACEHOLDER_BIRTH_DATES = [
    date(1899, 12, 31),
    date(1900, 1, 1),
]

placeholder_rows = transformed.filter(
    pl.col("birth_date").is_in(PLACEHOLDER_BIRTH_DATES)
).height

transformed = transformed.with_columns(
    pl.col("birth_date").replace(
        PLACEHOLDER_BIRTH_DATES,
        None,
    )
)

print(f"Placeholder birth dates converted to null: {placeholder_rows:,}")

In [ ]:
ISO_639_1_CODES = sorted(
    language.alpha_2
    for language in pycountry.languages
    if hasattr(language, "alpha_2")
)

transformed = transformed.with_columns(
    pl.col("preferred_language")
    .str.to_lowercase()
    .replace({"cz": "cs"})
)

In [ ]:
CONSENT_CAMPING_VALUES = {
    "J": True,
    "N": False,
}

transformed = transformed.with_columns(
    pl.col("consent_camping").replace_strict(
        CONSENT_CAMPING_VALUES
    )
)

print("Consent Camping after conversion")
print(
    transformed
    .group_by("consent_camping")
    .len(name="rows")
    .sort("rows", descending=True)
)

In [ ]:
GENDER_BY_SALUTATION = {
    "Mr.": "male",
    "Ms.": "female",
}

transformed = transformed.with_columns(
    pl.col("salutation")
    .replace_strict(GENDER_BY_SALUTATION, default=None)
    .alias("gender")
)

print("Gender derived from salutation")
print(
    transformed
    .group_by("gender")
    .len(name="rows")
    .sort("rows", descending=True)
)

In [ ]:
def normalize_phone(
    row: dict[str, str | None],
) -> str | None:
    value = row["phone"]

    if value is None:
        return None

    if not phonenumbers.is_possible_number_string(
        value,
        row["country"],
    ):
        return value

    return phonenumbers.format_number(
        phonenumbers.parse(
            value,
            row["country"],
        ),
        phonenumbers.PhoneNumberFormat.E164,
    )


phones_before = transformed.get_column("phone")

transformed = transformed.with_columns(
    pl.struct("phone", "country")
    .map_elements(
        normalize_phone,
        return_dtype=pl.String,
    )
    .alias("phone")
)

print(
    "Primary phones normalized to E.164: "
    f"{(phones_before != transformed['phone']).sum():,}"
)

In [ ]:
def phone_is_possible(row: dict[str, str | None]) -> bool:
    return row["phone"] is None or phonenumbers.is_possible_number_string(
        row["phone"],
        row["country"],
    )


phone_possible = transformed.select(
    pl.struct("phone", "country")
    .map_elements(
        phone_is_possible,
        return_dtype=pl.Boolean,
    )
    .alias("phone_possible")
).get_column("phone_possible")

transformed = transformed.with_columns(
    pl.when(phone_possible)
    .then(pl.col("phone"))
    .otherwise(None)
    .alias("phone")
)

print(
    "Invalid phones blanked (person kept): "
    f"{(~phone_possible).sum():,}"
)

## Person Account contract checks

Same schemas as the analysis notebook, reduced to what the export needs: the transformed
contract schema and the invalid/duplicate row indices that feed `contract_valid`.

In [ ]:
VALIDATION_DATE = date.today()
SALUTATIONS = ["Mr.", "Mrs.", "Ms.", "Dr.", "Prof."]
EMAIL_PATTERN = (
    r"^[0-9a-zA-Z]([-.\w]*[0-9a-zA-Z_+])*@"
    r"([0-9a-zA-Z][-\w]*[0-9a-zA-Z]\.)+"
    r"[a-zA-Z]{2,9}$"
)


def text_column(
    max_length: int,
    *,
    nullable: bool = True,
    required: bool = False,
) -> pa.Column:
    return pa.Column(
        pl.String,
        checks=pa.Check.str_length(max_value=max_length),
        nullable=nullable,
        required=required,
    )


def possible_phone(data: PolarsData) -> pl.LazyFrame:
    return data.lazyframe.select(
        pl.struct(data.key, "country").map_elements(
            lambda row: (
                row[data.key] is None
                or phonenumbers.is_possible_number_string(
                    row[data.key],
                    row["country"],
                )
            ),
            return_dtype=pl.Boolean,
        )
    )


person_account_source_schema = pa.DataFrameSchema(
    {
        "external_id": text_column(40, required=True),
        "cluster_id": text_column(40),
        "entra_external_id": text_column(255),
        "salutation": pa.Column(
            pl.String,
            checks=[
                pa.Check.str_length(max_value=40),
                pa.Check.isin(
                    SALUTATIONS,
                    name="valid_salutation",
                ),
            ],
            nullable=True,
            required=False,
        ),
        "first_name": text_column(40),
        "middle_name": text_column(40),
        "last_name": text_column(
            80,
            nullable=False,
            required=True,
        ),
        "birth_date": pa.Column(
            pl.Date,
            checks=[
                pa.Check.not_equal_to(
                    date(1899, 12, 31),
                    name="not_1899_placeholder",
                ),
                pa.Check.not_equal_to(
                    date(1900, 1, 1),
                    name="not_1900_placeholder",
                ),
                pa.Check.less_than_or_equal_to(
                    VALIDATION_DATE,
                    name="not_in_future",
                ),
            ],
            nullable=True,
            required=False,
        ),
        "birth_place": text_column(255),
        "gender": text_column(50),
        "email": pa.Column(
            pl.String,
            checks=[
                pa.Check.str_length(max_value=255),
                pa.Check.str_matches(
                    EMAIL_PATTERN,
                    name="valid_email",
                ),
            ],
            nullable=True,
            required=False,
        ),
        "phone": pa.Column(
            pl.String,
            checks=[
                pa.Check.str_length(max_value=50),
                pa.Check(
                    possible_phone,
                    name="possible_phone",
                ),
            ],
            nullable=True,
            required=False,
        ),
        "preferred_language": pa.Column(
            pl.String,
            checks=[
                pa.Check.str_length(max_value=10),
                pa.Check.isin(
                    ISO_639_1_CODES,
                    name="valid_iso_639_1",
                ),
            ],
            nullable=True,
            required=False,
        ),
        "nationality_country_code": text_column(10),
        "address": text_column(255),
        "postal_code": text_column(20),
        "city": text_column(40),
        "state": text_column(80),
        "country": text_column(80),
        "invest_customer": pa.Column(
            pl.Boolean,
            nullable=True,
            required=False,
        ),
        "investment_status": text_column(50),
        "investment_expiration_date": pa.Column(
            pl.Date,
            nullable=True,
            required=False,
        ),
    },
    name="person_account_source",
)

person_account_transformed_schema = (
    person_account_source_schema.add_columns(
        {
            "consent_camping": pa.Column(
                pl.Boolean,
                nullable=True,
                required=True,
            ),
            "exclude": pa.Column(
                pl.Boolean,
                nullable=False,
                required=True,
            ),
        }
    )
)

In [ ]:
FAILURE_CASES_SCHEMA = {
    "failure_case": pl.String,
    "schema_context": pl.String,
    "column": pl.String,
    "check": pl.String,
    "check_number": pl.Int64,
    "index": pl.Int64,
}


def collect_failure_cases(
    schema: pa.DataFrameSchema,
    frame: pl.DataFrame,
) -> pl.DataFrame:
    try:
        schema.validate(frame, lazy=True)
    except pa.errors.SchemaErrors as error:
        return error.failure_cases

    return pl.DataFrame(schema=FAILURE_CASES_SCHEMA)

In [ ]:
transformed_source_failures = collect_failure_cases(
    person_account_transformed_schema,
    transformed,
)

transformed_source_invalid_indices = (
    transformed_source_failures
    .get_column("index")
    .drop_nulls()
    .unique()
)

transformed_duplicate_rows = (
    transformed
    .with_row_index("index")
    .filter(
        ~pl.struct(transformed.columns).is_first_distinct()
    )
    .get_column("index")
)

rescue_indices = (
    pl.concat(
        [
            transformed_source_invalid_indices.to_frame(),
            transformed_duplicate_rows.to_frame(),
        ],
        how="vertical_relaxed",
    )
    .get_column("index")
    .unique()
)

print(f"Contract-invalid rows: {transformed_source_invalid_indices.len():,}")
print(f"Duplicate rows after first: {transformed_duplicate_rows.len():,}")
print(f"Combined rows requiring rescue: {rescue_indices.len():,}")

## Export

All rows with flags; filtering happens in MySQL.

Rows are deduplicated on the four matching fields (`first_name`, `last_name`, `email`,
`birth_date`), keeping the first occurrence.

`import_ready` implements the agreed upload scope (**Option 4 — DOB optional**):
first name, last name and valid email required, birth date optional,
`consent_camping` required — on top of `contract_valid` and not `exclude`.

In [ ]:
export = (
    transformed
    .with_row_index("row_index")
    .with_columns(
        contract_valid=~pl.col("row_index").cast(pl.Int64).is_in(rescue_indices)
    )
    .drop("row_index")
)

export_summary = pl.DataFrame(
    {
        "measure": [
            "total_rows",
            "exclude",
            "contract_valid",
            "contract_valid_and_not_excluded",
        ],
        "rows": [
            export.height,
            export.get_column("exclude").sum(),
            export.get_column("contract_valid").sum(),
            export.filter(
                pl.col("contract_valid") & ~pl.col("exclude")
            ).height,
        ],
    }
)
export_summary

In [ ]:
rows_before_dedup = export.height

export = export.unique(
    subset=["first_name", "last_name", "email", "birth_date"],
    keep="first",
    maintain_order=True,
)

print(f"Rows before deduplication: {rows_before_dedup:,}")
print(f"Duplicate rows removed: {rows_before_dedup - export.height:,}")
print(f"Rows after deduplication: {export.height:,}")

In [ ]:
export = export.with_columns(
    import_ready=(
        pl.col("contract_valid")
        & ~pl.col("exclude")
        & pl.col("first_name").is_not_null()
        & pl.col("last_name").is_not_null()
        & pl.col("email").is_not_null()
        & pl.col("consent_camping").fill_null(False)
    )
)

option4_before_consent = export.filter(
    pl.col("contract_valid")
    & ~pl.col("exclude")
    & pl.col("first_name").is_not_null()
    & pl.col("last_name").is_not_null()
    & pl.col("email").is_not_null()
).height

print(f"Option 4 rows before consent: {option4_before_consent:,}")
print(f"Option 4 import_ready (with consent): {export.get_column('import_ready').sum():,}")

In [ ]:
STANDARD_COLUMNS = [
    "sf_account_id",
    "sf_person_contact_id",
    "account_processed_at",
    "sf_loyalty_member_id",
    "sf_cp_email_id",
    "source",
    "source_origin",
    "external_id",
    "entra_external_id",
    "salutation",
    "first_name",
    "middle_name",
    "last_name",
    "birth_date",
    "birth_place",
    "gender",
    "email",
    "phone",
    "preferred_language",
    "nationality_country_code",
    "address",
    "postal_code",
    "city",
    "state",
    "country",
    "hotel_customer",
    "camping_customer",
    "residences_customer",
    "invest_customer",
    "primary_property_id",
    "loyalty_legacy_tier",
    "loyalty_legacy_number",
    "loyalty_points_balance",
    "loyalty_enrollment_date",
    "investment_status",
    "investment_expiration_date",
    "consent_central",
    "consent_camping",
    "consent_residences",
    "consent_invest",
]

# exclude -> data_issue on export: matches the investor staging convention and avoids
# having to quote a reserved word in every downstream query.
FLAG_RENAMES = {"exclude": "data_issue"}
FLAG_COLUMNS = ["data_issue", "contract_valid", "import_ready"]

empty = pl.lit(None, dtype=pl.String)

export = (
    export
    .with_columns(
        sf_account_id=empty,
        sf_person_contact_id=empty,
        account_processed_at=empty,
        sf_loyalty_member_id=empty,
        sf_cp_email_id=empty,
        source=pl.lit("camping"),
        source_origin=pl.lit("grubhof"),
        entra_external_id=empty,
        middle_name=empty,
        birth_place=empty,
        nationality_country_code=empty,
        state=empty,
        hotel_customer=pl.lit(0),
        camping_customer=pl.lit(1),
        residences_customer=pl.lit(0),
        invest_customer=pl.lit(0),
        primary_property_id=empty,
        loyalty_legacy_tier=empty,
        loyalty_legacy_number=empty,
        loyalty_points_balance=empty,
        loyalty_enrollment_date=empty,
        investment_status=empty,
        investment_expiration_date=empty,
        consent_central=pl.lit(0),
        consent_residences=pl.lit(0),
        consent_invest=pl.lit(0),
    )
    .with_columns(pl.col("consent_camping").cast(pl.Int8))
    .rename(FLAG_RENAMES)
    .with_columns(pl.col(FLAG_COLUMNS).cast(pl.Int8))
    .select(STANDARD_COLUMNS + FLAG_COLUMNS)
)

print(f"Standard format: {export.width} columns ({len(STANDARD_COLUMNS)} standard + {len(FLAG_COLUMNS)} flags)")
print(export.select(FLAG_COLUMNS).sum())

In [ ]:
export.write_csv(EXPORT_PATH)

print(f"Wrote {export.height:,} rows x {export.width} columns to {EXPORT_PATH.resolve()}")